In [ ]:
!pip install faker

import pandas as pd
import numpy as np
import uuid
import random
from faker import Faker
from datetime import datetime

fake = Faker()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 12.5 MB/s eta 0:00:00


In [ ]:
# ==========================
# BASIC SETTINGS
# ==========================

NUM_USERS = 10000
NUM_RECORDS = 80000
YEAR = 2025
BASE_FRAUD_RATE = 0.002
UPI_DAILY_LIMIT = 100000

# ==========================
# TIME PATTERN
# ==========================

PEAK_HOURS = list(range(10,14)) + list(range(17,22))
OFF_PEAK = list(set(range(24)) - set(PEAK_HOURS))

# ==========================
# BANKS
# ==========================

BANKS = [
    'State Bank of India', 'HDFC Bank', 'ICICI Bank',
    'Axis Bank', 'Punjab National Bank',
    'Bank of Baroda', 'Yes Bank',
    'Kotak Mahindra Bank', 'Canara Bank'
]

BANK_WEIGHTS = np.array([0.25,0.18,0.15,0.12,0.07,0.06,0.03,0.05,0.04])
BANK_WEIGHTS = BANK_WEIGHTS / BANK_WEIGHTS.sum()

# ==========================
# AGE GROUPS
# ==========================

AGE_GROUPS = ['18-25','26-35','36-45','46-60','60+']
AGE_WEIGHTS = np.array([0.18,0.35,0.22,0.15,0.10])
AGE_WEIGHTS = AGE_WEIGHTS / AGE_WEIGHTS.sum()

# ==========================
# STATES
# ==========================

STATES = [
    'Maharashtra','Uttar Pradesh','Tamil Nadu','Karnataka',
    'West Bengal','Gujarat','Rajasthan','Bihar','Odisha','Delhi'
]

STATE_WEIGHTS = np.array([0.15,0.12,0.10,0.08,0.08,0.07,0.06,0.06,0.05,0.03])
STATE_WEIGHTS = STATE_WEIGHTS / STATE_WEIGHTS.sum()

# ==========================
# MERCHANT CATEGORIES
# ==========================

MERCHANT_CATEGORIES = [
    'Grocery','Food Delivery','Bill Payment','Recharge',
    'Travel','Entertainment','Healthcare','Education','Others'
]

MERCHANT_WEIGHTS = np.array([0.20,0.15,0.18,0.10,0.05,0.07,0.05,0.03,0.17])
MERCHANT_WEIGHTS = MERCHANT_WEIGHTS / MERCHANT_WEIGHTS.sum()

# ==========================
# DEVICE TYPES
# ==========================

DEVICE_TYPES = ['Android','iOS','Web']
DEVICE_WEIGHTS = np.array([0.80,0.18,0.02])
DEVICE_WEIGHTS = DEVICE_WEIGHTS / DEVICE_WEIGHTS.sum()

UPI_SUFFIXES = ['@okaxis','@ibl','@oksbi','@ybl','@upi']


In [ ]:
users = []

for _ in range(NUM_USERS):   # <-- NOT while
    name = fake.name()
    clean_name = "".join(name.lower().split())
    upi_id = clean_name + random.choice(UPI_SUFFIXES)

    balance = np.random.uniform(20000, 200000)

    users.append({
        "name": name,
        "upi_id": upi_id,
        "bank": np.random.choice(BANKS, p=BANK_WEIGHTS),
        "age_group": np.random.choice(AGE_GROUPS, p=AGE_WEIGHTS),
        "state": np.random.choice(STATES, p=STATE_WEIGHTS),
        "avg_amount": np.random.uniform(300, 5000),
        "account_balance": balance
    })

print("Users created:", len(users))


Users created: 10000


In [ ]:
def generate_timestamp():
    start = datetime(YEAR,1,1)
    end = datetime(YEAR,12,31)

    random_date = start + (end - start) * random.random()

    if random.random() < 0.75:
        hour = random.choice(PEAK_HOURS)
    else:
        hour = random.choice(OFF_PEAK)

    return random_date.replace(
        hour=hour,
        minute=random.randint(0,59),
        second=random.randint(0,59)
    )


In [ ]:
data = []

while len(data) < NUM_RECORDS:

    sender = random.choice(users)

    # Skip if balance too low
    if sender["account_balance"] < 100:
        continue

    txn_id = str(uuid.uuid4())
    timestamp = generate_timestamp()

    hour_of_day = timestamp.hour
    day_of_week = timestamp.strftime('%A')
    is_weekend = 1 if day_of_week in ['Saturday','Sunday'] else 0

    # =====================================
    # AMOUNT GENERATION (5% anomaly spike)
    # =====================================

    if random.random() < 0.95:
        # Normal behavior
        amount = np.random.normal(sender["avg_amount"], sender["avg_amount"] * 0.4)
    else:
        # Rare anomaly spike
        amount = sender["avg_amount"] * np.random.uniform(3, 6)

    amount = abs(amount)

    # Cap to balance
    amount = min(amount, sender["account_balance"])

    # Apply UPI daily limit
    amount = min(amount, UPI_DAILY_LIMIT)

    amount = round(amount, 2)

    transfer_ratio = amount / sender["account_balance"]

    # =====================================
    # RECEIVER GENERATION
    # =====================================

    receiver_is_merchant = random.random() < 0.30

    if receiver_is_merchant:
        receiver_name = fake.company()
        clean_receiver = "".join(receiver_name.lower().split())
        receiver_upi_id = clean_receiver + random.choice(UPI_SUFFIXES)
        merchant_category = np.random.choice(MERCHANT_CATEGORIES, p=MERCHANT_WEIGHTS)
        txn_type = 'P2M'
    else:
        receiver = random.choice(users)
        receiver_name = receiver["name"]
        receiver_upi_id = receiver["upi_id"]
        merchant_category = 'Peer'
        txn_type = 'P2P'

    # =====================================
    # CALIBRATED FRAUD LOGIC (Realistic)
    # =====================================

    fraud_flag = 0

    # Hard triggers (rare extreme events)
    if transfer_ratio > 0.95:
        fraud_flag = 1

    elif amount > 6 * sender["avg_amount"]:
        fraud_flag = 1

    else:
        fraud_probability = BASE_FRAUD_RATE  # 0.2%

        if amount > 3 * sender["avg_amount"]:
            fraud_probability += 0.03

        if transfer_ratio > 0.75:
            fraud_probability += 0.04

        if hour_of_day >= 23 or hour_of_day <= 4:
            fraud_probability += 0.005

        fraud_flag = 1 if random.random() < fraud_probability else 0

    # Deduct balance
    sender["account_balance"] -= amount

    record = {
        'transaction_id': txn_id,
        'timestamp': timestamp,
        'hour_of_day': hour_of_day,
        'day_of_week': day_of_week,
        'is_weekend': is_weekend,
        'amount': amount,
        'transfer_ratio': round(transfer_ratio, 3),
        'txn_type': txn_type,
        'merchant_category': merchant_category,

        'sender_name': sender["name"],
        'sender_upi_id': sender["upi_id"],
        'sender_bank': sender["bank"],
        'sender_age_group': sender["age_group"],
        'sender_state': sender["state"],

        'receiver_name': receiver_name,
        'receiver_upi_id': receiver_upi_id,
        'receiver_bank': np.random.choice(BANKS, p=BANK_WEIGHTS),

        'device_type': np.random.choice(DEVICE_TYPES, p=DEVICE_WEIGHTS),
        'network_type': random.choice(['4G','5G','WiFi']),

        'fraud_flag': fraud_flag
    }

    data.append(record)

df = pd.DataFrame(data)

print("Dataset Shape:", df.shape)
print("Fraud Rate:", df['fraud_flag'].mean())
df.head()


Dataset Shape: (80000, 20)
Fraud Rate: 0.0109625


,transaction_id,timestamp,hour_of_day,day_of_week,is_weekend,amount,transfer_ratio,txn_type,merchant_category,sender_name,sender_upi_id,sender_bank,sender_age_group,sender_state,receiver_name,receiver_upi_id,receiver_bank,device_type,network_type,fraud_flag
0,7048b9d3-0135-4d2f-af96-a6038c531a50,2025-11-27 22:28:11.303193,22,Thursday,0,5652.02,0.123,P2P,Peer,Michael Young,michaelyoung@upi,ICICI Bank,60+,Karnataka,Michael Shields,michaelshields@upi,State Bank of India,Android,WiFi,0
1,57c1acf8-c3a9-4d6e-8c2b-e3a6cda0d8ff,2025-03-14 11:32:02.891216,11,Friday,0,1707.79,0.034,P2P,Peer,Heather Snyder,heathersnyder@okaxis,State Bank of India,26-35,Gujarat,Jasmine Mcdonald,jasminemcdonald@okaxis,Axis Bank,Web,5G,0
2,24993065-846d-483d-b034-bb8e74f244c4,2025-07-19 10:49:52.084414,10,Saturday,1,2297.10,0.015,P2P,Peer,Patty Rush,pattyrush@upi,HDFC Bank,46-60,Maharashtra,James Hunter,jameshunter@oksbi,HDFC Bank,Android,4G,0
3,301e9418-95fd-4e33-a98e-c724777d039c,2025-12-03 21:03:36.766730,21,Wednesday,0,2716.05,0.032,P2M,Food Delivery,Kristy Skinner,kristyskinner@upi,HDFC Bank,18-25,Tamil Nadu,"Webb, Barrett and Clark","webb,barrettandclark@ybl",ICICI Bank,Android,4G,0
4,f5dcda42-27b2-470f-82c1-3dca8f6b07a2,2025-11-25 20:27:16.945209,20,Tuesday,0,1116.71,0.010,P2P,Peer,Morgan Harmon,morganharmon@ibl,HDFC Bank,36-45,Bihar,Ryan Ryan DDS,ryanryandds@oksbi,Punjab National Bank,Android,5G,0


In [ ]:
df['fraud_flag'].value_counts()


,count
fraud_flag,
0,79123
1,877


In [ ]:
df.to_csv("synthetic_upi_transactions.csv", index=False)

from google.colab import files
files.download("synthetic_upi_transactions.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>